# Lab Assignment 2 - Part C: Naive Bayes for Spam Detection
Please refer to the `README.pdf` for full laboratory instructions.


## Problem Statement
In this part, you will implement a **Naive Bayes classifier** for spam email detection using the Spambase dataset.

### Dataset Description
The Spambase dataset contains 4601 email samples with 57 features:
- **Features 1-48**: Word frequencies (percentage of words matching specific words)
- **Features 49-54**: Character frequencies (`;`, `(`, `[`, `!`, `$`, `#`)
- **Features 55-57**: Capital letter statistics
- **Label**: 1 = spam, 0 = not spam

### Your Tasks
1. **Implement Gaussian Naive Bayes** from scratch
2. **Train and evaluate** your classifier (accuracy, precision, recall, F1-score)
3. **Feature analysis**: Identify top discriminative features
4. **Discussion**: Why does Naive Bayes work for spam detection?


## Setup


In [ ]:
%pip install ucimlrepo


In [ ]:
# Library declarations
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo


## Load the Spambase Dataset


In [ ]:
# Fetch Spambase dataset from UCI ML Repository
spambase = fetch_ucirepo(id=94)

# Get features and labels
X = spambase.data.features.values
y = spambase.data.targets.values.ravel()

print(f"Dataset shape: {X.shape}")
print(f"Number of spam emails: {np.sum(y == 1)}")
print(f"Number of non-spam emails: {np.sum(y == 0)}")
print(f"\nFeature names: {list(spambase.data.features.columns[:10])}...")  # First 10 features


In [ ]:
# Split data into training (80%) and testing (20%) sets
def train_test_split(X, y, test_size=0.2, random_state=42):
    """
    Split data into training and testing sets.
    """
    np.random.seed(random_state)
    n_samples = len(y)
    indices = np.random.permutation(n_samples)
    test_size = int(n_samples * test_size)
    
    test_indices = indices[:test_size]
    train_indices = indices[test_size:]
    
    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")


## Task 1: Implement Gaussian Naive Bayes
Implement a Gaussian Naive Bayes classifier from scratch.

**Key formulas:**
- Class prior: $P(C) = \frac{N_C}{N}$
- Gaussian likelihood: $P(x_i|C) = \frac{1}{\sqrt{2\pi\sigma_{i,C}^2}} \exp\left(-\frac{(x_i - \mu_{i,C})^2}{2\sigma_{i,C}^2}\right)$
- Use **log-probabilities** to avoid numerical underflow!


In [ ]:
class GaussianNaiveBayes:
    """
    Gaussian Naive Bayes classifier implementation.
    """
    
    def __init__(self):
        self.classes = None
        self.priors = None      # P(C) for each class
        self.means = None       # Mean of each feature per class
        self.variances = None   # Variance of each feature per class
    
    def fit(self, X, y):
        """
        Train the Naive Bayes classifier.
        
        Parameters:
        -----------
        X : numpy array of shape (n_samples, n_features)
        y : numpy array of shape (n_samples,)
        """
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        
        # Calculate class priors P(C)
        self.priors = np.zeros(n_classes)
        for i, c in enumerate(self.classes):
            self.priors[i] = np.sum(y == c) / n_samples
        
        # Calculate mean and variance for each feature per class
        self.means = np.zeros((n_classes, n_features))
        self.variances = np.zeros((n_classes, n_features))
        
        epsilon = 1e-9  # Small value to avoid division by zero
        
        for i, c in enumerate(self.classes):
            X_c = X[y == c]
            self.means[i, :] = X_c.mean(axis=0)
            self.variances[i, :] = X_c.var(axis=0) + epsilon
    
    def _gaussian_log_likelihood(self, x, mean, var):
        """
        Calculate log of Gaussian probability density.
        
        Returns:
        --------
        log_likelihood : float or array
        """
        # log P(x|C) = -0.5 * log(2*pi*var) - 0.5 * (x - mean)^2 / var
        return -0.5 * np.log(2 * np.pi * var) - 0.5 * ((x - mean) ** 2) / var
    
    def predict(self, X):
        """
        Predict class labels for samples in X.
        
        Parameters:
        -----------
        X : numpy array of shape (n_samples, n_features)
        
        Returns:
        --------
        predictions : numpy array of shape (n_samples,)
        """
        n_samples = X.shape[0]
        n_classes = len(self.classes)
        log_probs = np.zeros((n_samples, n_classes))
        
        for i in range(n_classes):
            # Start with log prior
            log_prior = np.log(self.priors[i])
            
            # Sum log likelihoods across all features
            log_likelihood = np.sum(
                self._gaussian_log_likelihood(X, self.means[i], self.variances[i]),
                axis=1
            )
            
            log_probs[:, i] = log_prior + log_likelihood
        
        # Return class with highest log probability
        return self.classes[np.argmax(log_probs, axis=1)]
    
    def predict_proba(self, X):
        """
        Return probability estimates for samples using softmax on log posteriors.
        
        Returns:
        --------
        probabilities : numpy array of shape (n_samples, n_classes)
        """
        n_samples = X.shape[0]
        n_classes = len(self.classes)
        log_probs = np.zeros((n_samples, n_classes))
        
        for i in range(n_classes):
            log_prior = np.log(self.priors[i])
            log_likelihood = np.sum(
                self._gaussian_log_likelihood(X, self.means[i], self.variances[i]),
                axis=1
            )
            log_probs[:, i] = log_prior + log_likelihood
        
        # Softmax for numerical stability
        log_probs_stable = log_probs - np.max(log_probs, axis=1, keepdims=True)
        probs = np.exp(log_probs_stable)
        probs = probs / np.sum(probs, axis=1, keepdims=True)
        
        return probs

## Task 2: Train and Evaluate
Train your classifier and compute evaluation metrics.


In [ ]:
def compute_metrics(y_true, y_pred):
    """
    Compute classification metrics.
    
    Returns:
    --------
    accuracy, precision, recall, f1_score : floats
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return accuracy, precision, recall, f1_score


def confusion_matrix(y_true, y_pred):
    """
    Create confusion matrix.
    
    Returns:
    --------
    matrix : numpy array of shape (2, 2)
        [[TN, FP], [FN, TP]]
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    
    return np.array([[TN, FP], [FN, TP]])

## Task 3: Feature Analysis
Identify the most discriminative features for spam detection.


In [ ]:
# Train the classifier
nb_classifier = GaussianNaiveBayes()
nb_classifier.fit(X_train, y_train)

# Make predictions
y_pred = nb_classifier.predict(X_test)

# Compute and print metrics
accuracy, precision, recall, f1 = compute_metrics(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# Create and display confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(cm)

In [ ]:
# Identify top 5 most discriminative features
# Use normalized mean difference (difference of class means / pooled std)
feature_names = list(spambase.data.features.columns)

discriminative_scores = []
for i in range(X.shape[1]):
    mean_spam = nb_classifier.means[1, i]
    mean_not_spam = nb_classifier.means[0, i]
    
    # Pooled standard deviation
    var_spam = nb_classifier.variances[1, i]
    var_not_spam = nb_classifier.variances[0, i]
    pooled_std = np.sqrt((var_spam + var_not_spam) / 2)
    
    # Normalized difference of means
    score = np.abs(mean_spam - mean_not_spam) / pooled_std if pooled_std > 0 else 0
    discriminative_scores.append(score)

discriminative_scores = np.array(discriminative_scores)

# Get top 5 features
top_5_indices = np.argsort(discriminative_scores)[::-1][:5]

print("Top 5 most discriminative features:")
for rank, idx in enumerate(top_5_indices, 1):
    print(f"  {rank}. {feature_names[idx]}: score = {discriminative_scores[idx]:.4f}")

In [ ]:
# Visualize distributions of top features for spam vs non-spam
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, idx in enumerate(top_5_indices[:5]):
    ax = axes[i]
    ax.hist(X[y == 0, idx], bins=30, alpha=0.5, label='Not Spam', density=True)
    ax.hist(X[y == 1, idx], bins=30, alpha=0.5, label='Spam', density=True)
    ax.set_title(f'{feature_names[idx]}')
    ax.legend()

# Hide the 6th subplot
axes[5].set_visible(False)

plt.tight_layout()
plt.show()

## Summary and Discussion

### Results Table
| Metric | Value |
|--------|-------|
| Accuracy | 0.8217 |
| Precision | 0.7233 |
| Recall | 0.9385 |
| F1-Score | 0.8170 |

### Top 5 Discriminative Features
1. word_freq_your
2. word_freq_remove
3. word_freq_000
4. char_freq_$
5. word_freq_you

### Discussion

1. **Why is Naive Bayes effective for spam detection despite the independence assumption?**
   Naive Bayes works well for spam detection because the classifier only needs to get the correct ranking of class posteriors (spam vs not spam), not the exact probabilities. Even though features like word frequencies are not truly independent, the model can still identify the right class when strong discriminative signals are present. Spam emails tend to have distinctive patterns (e.g., high frequency of words like "remove", "your", "$") that provide strong enough signal individually that the independence assumption does not significantly hurt classification accuracy.

2. **What are the limitations of your implementation?**
   The Gaussian assumption may not fit well for word frequency features, which are often zero-heavy and right-skewed rather than normally distributed. Many features have a large spike at zero with a long tail, which the Gaussian model handles poorly. Additionally, the model treats all features equally even though some may be noisy and contribute little useful information.

3. **How could you improve the classifier?**
   Using a different distribution (e.g., Bernoulli or Multinomial Naive Bayes) may better suit the discrete/count nature of word frequencies. Feature selection to remove low-information features could also help. Tuning the variance smoothing parameter or applying log-transforms to heavy-tailed features could improve the Gaussian fit.

4. **What did I learn from this exercise?**
   I learned that Naive Bayes is a simple yet surprisingly effective baseline for text classification tasks like spam detection. The use of log-probabilities is essential to avoid numerical underflow when multiplying many small probabilities. I also saw how analyzing the most discriminative features gives interpretable insights into what the model is learning -- the top features align with intuitive spam signals like dollar signs and words like "remove" and "your".